This notebook is intended to run at USDF and can be used to plot the thermal gradients across M1M3 along the X, Y, Z, and Radial axes. 

It has the following T&S package dependencies:
- https://github.com/lsst-ts/ts_m1m3_utils
- https://github.com/lsst-ts/ts_xml
- https://github.com/lsst-ts/ts_utils 
- https://github.com/lsst-ts/ts_salobj 

which can be installed in a users local environment by cloing the repositories and sourcing them in the .user_setups file with something like "setup -j ts_m1m3_utils -r /home/e/edennihy/u/repos/ts_m1m3_utils". 

It also has the following external dependencies which should be installed by a package manager (e.g. pip) into the users environment:
- orjson
- authlib
- pyside6

In [ ]:
from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import *
from astropy.time import Time, TimeDelta

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib import cm

%matplotlib widget 

# make EFD client
from lsst_efd_client import EfdClient
efd = EfdClient('usdf_efd')



In [ ]:
ThermocoupleAnalysis?

In [ ]:
time_start = Time('2025-11-10 00:19:00')
time_end = Time('2025-11-10 08:09:00')

thermocouples = ThermocoupleAnalysis(efd)
await thermocouples.load(time_start, time_end, time_bin=30)

In [ ]:
print(type(thermocouples.xyz_r_gradients))

In [ ]:
thermocouples.xyz_r_gradients

In [ ]:
gradients = thermocouples.xyz_r_gradients

In [ ]:
grad_times = pd.to_datetime(gradients.index, format='ISO8601', utc=True).astype('int64')
grad_times -= grad_times[0]
grad_times /=1E9

In [ ]:
gradients['x_gradient'].values[100:110]

In [ ]:
grad_times

In [ ]:
%matplotlib inline
gradients['x_gradient']

In [ ]:
plt.plot(
    thermocouples.xyz_r_gradients.x_gradient * 8.4,
    label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
    marker="o", markersize=3, markevery=50, zorder=3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# --- Palettes: same hue family within groups, distinct shades per line ---
blues   = cm.get_cmap('Blues')
oranges = cm.get_cmap('Oranges')

# Line colors (distinct, but clearly grouped)
c_x = blues(0.75)     # darker blue
c_y = blues(0.55)     # lighter blue
c_r = oranges(0.75)   # darker orange
c_z = oranges(0.55)   # lighter orange

# Band colors (very light tint of the group hue)
band_xy = blues(0.25)
band_rz = oranges(0.25)

# --- Shaded operational bands (put behind data) ---
ax.axhspan(-0.4, 0.4, facecolor=band_xy, alpha=0.15, zorder=0)
ax.axhspan(-0.1, 0.1, facecolor=band_rz, alpha=0.20, zorder=0)
ax.axhline(0.4,  color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.4, color=blues(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(0.1,  color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)
ax.axhline(-0.1, color=oranges(0.65), linestyle="-", linewidth=1, alpha=0.3)

# --- Data lines: distinct per series with styles/markers ---
ln_x, = ax.plot(
    thermocouples.xyz_r_gradients.x_gradient * 8.4,
    label="X (×8.4)", color=c_x, linewidth=2.0, linestyle="-",
    marker="o", markersize=3, markevery=50, zorder=3
)
ln_y, = ax.plot(
    thermocouples.xyz_r_gradients.y_gradient * 8.4,
    label="Y (×8.4)", color=c_y, linewidth=2.0, linestyle="--",
    marker="s", markersize=3, markevery=50, zorder=3
)
ln_r, = ax.plot(
    thermocouples.xyz_r_gradients.radial_gradient * 4.2,
    label="Radial (×4.2)", color=c_r, linewidth=2.0, linestyle="-",
    marker="^", markersize=3, markevery=50, zorder=4
)
ln_z, = ax.plot(
    thermocouples.xyz_r_gradients.z_gradient,
    label="Z", color=c_z, linewidth=2.0, linestyle="--",
    marker="D", markersize=3, markevery=50, zorder=4
)

# --- Time formatting (if index is datetime) ---
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

# --- Colored labels on the right, matching each band hue ---
ax.annotate("X/Y operational limit ±0.4",
            xy=(1, 0.25), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=blues(0.6), fontsize=9)
ax.annotate("Radial/Z operational limit ±0.1",
            xy=(1, 0.0), xycoords=("axes fraction", "data"),
            xytext=(6, 0), textcoords="offset points",
            ha="left", va="bottom", color=oranges(0.6), fontsize=9)

# --- Legends: one for data lines, one for bands (with matching colors) ---
data_leg = ax.legend(handles=[ln_x, ln_y, ln_r, ln_z],
                     loc="upper left", frameon=True)
ax.add_artist(data_leg)

band_handles = [
    Patch(facecolor=band_xy, alpha=0.25, label="X/Y limit band (±0.4)"),
    Patch(facecolor=band_rz, alpha=0.25, label="Radial/Z limit band (±0.1)"),
]
ax.legend(handles=band_handles, title="Operational Limits",
          loc="lower left", frameon=True)

# --- Labels, grid, cosmetics ---

# --- Gather all y-data across plotted series ---
ydata = np.concatenate([
    ln_x.get_ydata(),
    ln_y.get_ydata(),
    ln_r.get_ydata(),
    ln_z.get_ydata(),
])

# --- Compute the extrema safely ---
ymin = np.nanmin(ydata)
ymax = np.nanmax(ydata)

# --- Apply conditional limits ---
lower = ymin if ymin < -0.8 else -0.8
upper = ymax if ymax >  0.8 else  0.8
ax.set_ylim(lower, upper)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Thermal Gradient (deg C)")
ax.set_title("M1M3 Thermal Gradients Across the Mirror")
ax.grid(True, linestyle=":", alpha=0.6)
# Show minutes in the x-axis labels (and full date so it’s unambiguous)
locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- Compute the weighted metric (normalized by each group's operational limit) ---

weighted_gradients = (
      0.3 * np.abs(thermocouples.xyz_r_gradients["z_gradient"]      / 0.1)
    + 0.3 * np.abs(thermocouples.xyz_r_gradients["radial_gradient"]*4.2 / 0.1)
    + 0.2 * np.abs(thermocouples.xyz_r_gradients["y_gradient"]*8.4      / 0.4)
    + 0.2 * np.abs(thermocouples.xyz_r_gradients["x_gradient"]*8.4      / 0.4)
)

thermocouples.xyz_r_gradients = thermocouples.xyz_r_gradients.assign(weighted=weighted_gradients)

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(thermocouples.xyz_r_gradients["weighted"].index, thermocouples.xyz_r_gradients["weighted"].values, label="Weighted Gradient Score", lw=1.6)

# Reference line at 1.0 (≈ operational limit boundary after normalization & weights)
ax.axhline(1.0, ls="--", lw=1.2, color="k", alpha=0.7, label="Reference = 1.0")

# Lightly shade regions where the weighted index exceeds 1.0
ax.fill_between(
    thermocouples.xyz_r_gradients["weighted"].index,
    thermocouples.xyz_r_gradients["weighted"].values,
    1.0,
    where=(thermocouples.xyz_r_gradients["weighted"].values >= 1.0),
    alpha=0.15,
    step=None
)

# Formatting
ax.set_ylabel("Weighted Gradient Score")
ax.set_xlabel("Time (UTC)")

# Show minutes in the x-axis labels (and full date so it’s unambiguous)
locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
ax.xaxis.set_major_locator(locator)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

ax.grid(True, linestyle=":", alpha=0.6)
ax.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="0.85")

ax.set_title("Weighted M1M3 Temperature-Gradient Score > 1.0 indicates strong thermal gradients")
plt.tight_layout()
plt.show()